In [32]:
%pip install tokenizers==0.22.0 transformers==5.3.0 huggingface-hub==1.3.0 torch==2.4 --no-deps
%pip install regex numpy requests filelock packaging safetensors

ERROR: Could not find a version that satisfies the requirement torch==2.4 (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
ERROR: No matching distribution found for torch==2.4
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import tokenizers
print(tokenizers.__version__)

0.21.0


In [21]:
with open("./droiddetect_tokenizer/tokenizer.json") as f:
    content = f.read()

# Find what's around character 251502
print(content[251400:251600])

   "Ġ97": 10694,
      "800": 10695,
      "Ġinflammation": 10696,
      "Car": 10697,
      "Ġunsigned": 10698,
      "Ġroof": 10699,
      "town": 10700,
      "ĠNetwork": 10701,
      "Ġelevated": 


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Building blocks ──────────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * self.weight

class Attention(nn.Module):
    def __init__(self, hidden_size, num_heads=16):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.Wqkv = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        self.Wo = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        qkv = self.Wqkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.num_heads, self.head_dim).transpose(1, 2) for t in qkv]
        scores = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5
        
        if attention_mask is not None:
            mask = attention_mask[:, None, None, :].float()
            scores = scores + (1.0 - mask) * -1e9
            
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.Wo(out)

class SwiGLUMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size=2624):
        super().__init__()
        self.Wi = nn.Linear(hidden_size, intermediate_size * 2, bias=False)
        self.Wo = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        a, b = self.Wi(x).chunk(2, dim=-1)
        return self.Wo(F.silu(a) * b)

class TransformerLayer(nn.Module):
    def __init__(self, hidden_size, first_layer=False):
        super().__init__()
        if not first_layer:
            self.attn_norm = RMSNorm(hidden_size)
        self.attn = Attention(hidden_size)
        self.mlp_norm = RMSNorm(hidden_size)
        self.mlp     = SwiGLUMLP(hidden_size, intermediate_size=2624)
        self.first_layer = first_layer

    def forward(self, x, attention_mask=None):
        normed = x if self.first_layer else self.attn_norm(x)
        x = x + self.attn(normed, attention_mask)
        x = x + self.mlp(self.mlp_norm(x))
        return x

class EncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state
        
class TextEncoder(nn.Module): # :white_check_mark: class name restored
    def __init__(self, vocab_size=50368, hidden_size=1024, num_layers=28):
        super().__init__()
        self.embeddings = nn.ModuleDict({
            "tok_embeddings": nn.Embedding(vocab_size, hidden_size),
            "norm": RMSNorm(hidden_size),
        })
        self.layers = nn.ModuleList([
            TransformerLayer(hidden_size, first_layer=(i == 0))
            for i in range(num_layers)
        ])
        self.final_norm = RMSNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        x = self.embeddings["tok_embeddings"](input_ids)
        x = self.embeddings["norm"](x)
        for layer in self.layers:
            x = layer(x, attention_mask)
        x = self.final_norm(x)
        # :white_check_mark: return object with .last_hidden_state to match TLModel's original usage
       
        return EncoderOutput(x)

# ── Top-level model ──────────────────────────────────────────────────────────

class TLModel(nn.Module):
    def __init__(self, hidden_size=1024, projection_dim=256, num_classes=4): # :white_check_mark: 256 not 128
        super().__init__() # :white_check_mark: was missing
        self.text_encoder = TextEncoder(hidden_size=hidden_size)
        self.text_projection = nn.Linear(hidden_size, projection_dim) # :white_check_mark: 1024 not 768
        self.classifier = nn.Linear(projection_dim, num_classes)

    def forward(self, input_ids=None, attention_mask=None):
        encoder_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        x = encoder_out.last_hidden_state.mean(dim=1) # :white_check_mark: mean pooling (not CLS)
        x = F.relu(self.text_projection(x))  # :white_check_mark: relu matches original
        logits = self.classifier(x)
        return logits


In [23]:
import tokenizers
print(tokenizers.__version__)
import sys
print(sys.executable)

0.21.0
/Users/magnus/iCloud Drive (Archive)/NTNU/Semester 10/TDT4900/pgs_v1/.venvipynb/bin/python


In [24]:
with open("./droiddetect_tokenizer/tokenizer.json") as f:
    content = f.read()

print(f"Total length: {len(content)}")

# Find the exact problematic spot
print(content[251450:251600])

Total length: 3494741
mmation": 10696,
      "Car": 10697,
      "Ġunsigned": 10698,
      "Ġroof": 10699,
      "town": 10700,
      "ĠNetwork": 10701,
      "Ġelevated": 


In [30]:
import torch
print(torch.__version__)

2.2.2


In [33]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer

# 1. Load tokenizer directly - no transformers needed
tokenizer = Tokenizer.from_file("./droiddetect_tokenizer/tokenizer.json")

# 2. Load model
model = TLModel()
model.load_state_dict(torch.load("droiddetect_weights.pt", map_location="cpu"))
model.eval()

# 3. Classify
id2label = {
    0: "HUMAN_GENERATED",
    1: "MACHINE_GENERATED",
    2: "MACHINE_REFINED",
    3: "MACHINE_GENERATED_ADVERSARIAL",
}

def classify(text):
    # Tokenize directly with tokenizers library, no transformers
    encoding = tokenizer.encode(text)
    input_ids = torch.tensor([encoding.ids])
    attention_mask = torch.tensor([encoding.attention_mask])

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)

    probs = F.softmax(logits, dim=-1)[0]
    predicted = probs.argmax().item()
    print(f"Prediction : {id2label[predicted]}")
    print(f"Confidence : {probs[predicted]:.1%}")
    print()
    for i, p in enumerate(probs):
        bar = "█" * int(p * 40)
        print(f"  {id2label[i]:<32} {p:.4f}  {bar}")

In [37]:
ai_generated_python_code = """
import React, { useState, useEffect } from "react";
import { MapContainer, Marker, Popup, TileLayer } from "react-leaflet";
import "./map.css";
import SearchBar from "material-ui-search-bar";

export default function MyMap() {

    const [zoomPos, setZoomPos] = useState([63.4428578, 10.4285791])
    const [pos, setPos] = useState([[63.442858, 10.428579], [63.442850, 10.42857]]);
    const [tagId, setTagId] = useState()

    return (
        <div>
            <div>
                <SearchBar
                    style={{ marginBottom: "5vh", marginTop: "5vh" }}
                    onChange={(newValue) => console.log(newValue)}
                    onRequestSearch={(value) => setTagId(value)}
                />
            </div>
            <MapContainer style={{ height: "70vh", width: "100%" }} center={zoomPos} zoom={19} scrollWheelZoom={false}>
                <TileLayer
                    attribution='&copy; <a href="http://osm.org/copyright">OpenStreetMap</a> contributors'
                    url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
                />
                {pos.map(s => {
                    return (
                        <Marker key={s} position={s}>
                            <Popup>
                                {s}
                            </Popup>
                        </Marker>)
                })}
            </MapContainer>

        </div>
    );
}

"""

classify(ai_generated_python_code)

Prediction : HUMAN_GENERATED
Confidence : 72.0%

  HUMAN_GENERATED                  0.7201  ████████████████████████████
  MACHINE_GENERATED                0.0189  
  MACHINE_REFINED                  0.0385  █
  MACHINE_GENERATED_ADVERSARIAL    0.2225  ████████
